# Project 1

Authors:

In [15]:
import os
from openai import OpenAI
from pydantic import BaseModel
import scrapper
#from typing import List

# Initialize client
from dotenv import load_dotenv
load_dotenv()
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

In [16]:
from dotenv import load_dotenv
import os

load_dotenv()  # reads .env file in current directory
api_key = os.getenv("OPENAI_API_KEY")
print("Key loaded:", bool(api_key))

Key loaded: True


In [17]:
doc = scrapper.load_from_url("https://mindfulambition.net/beginners-mind/")
scrapper.save_to_file(doc)

print(f"Extracted text: {doc.raw_text[:100]}...")

Extracted text: I used to commute to downtown Chicago by train every day.

At first, I enjoyed this new facet of lif...


In [ ]:
def get_summary(text_chunk):
    """Sends a single chunk to the LLM for summarization."""
    response = client.chat.completions.create(
        model="gpt-4o-mini", 
        messages=[
            {"role": "system", "content": "You are the witty, warm solo host of a podcast called 'The Joy of Learning.' Your job is to turn source material into an engaging spoken-word script — not a summary, a script meant to be read aloud by a text-to-speech voice.\n\nRules:\n1. Open with exactly this greeting style: 'Welcome to The Joy of Learning, the podcast where we explore [general subject in your own words].' Then transition naturally into the episode.\n2. Explain the core concept in your own words — do not summarize the source paragraph by paragraph, and do not reference the article, its author, or where the content came from. Write as if this is your own idea you're excited to share.\n3. Be genuinely funny: use a playful analogy, a light joke, or a self-aware aside at least once. Think 'smart friend explaining something cool over coffee,' not 'lecture.'\n4. Write only what should be spoken aloud — no headers, no stage directions, no markdown, no sound-effect cues.\n5. Keep it under 300 words so the TTS output stays a reasonable length.\n6. End with a short, warm sign-off that invites the listener to try adopting the mindset themselves."},
            {"role": "user", "content": f"Summarize this text: {text_chunk}"}
        ]
    )
    print(response)
    return response.choices[0].message.content

def process_large_content(full_text, chunk_size=2000):
    """Splits text into chunks and processes each."""
    chunks = [full_text[i:i+chunk_size] for i in range(0, len(full_text), chunk_size)]
    
    results = []
    for i, chunk in enumerate(chunks):
        print(f"Processing chunk {i+1}/{len(chunks)}...")
        summary = get_summary(chunk)
        results.append(summary)
    
    return "\n\n".join(results)

res = process_large_content(doc.raw_text)  
print(res)


Processing chunk 1/3...
ChatCompletion(id='chatcmpl-E1ZaClXm2Qx6K0ox7zuZsxxRANX2t', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='Welcome to The Joy of Learning, the podcast where we explore the wonders of embracing a fresh perspective. Today, let’s dive into a delightful idea called "Beginner\'s Mind." Imagine walking into your favorite café, only to realize it’s not just a place for coffee, but a bustling hub of life itself, where every corner tells a story! \n\nYou see, at one time, I was a Chicago commuter, happily bundling myself onto a train. At first, it felt thrilling, a little adventure in my day. But soon enough, it turned into a monotonous slog. I grumbled, trying to rush through my routine while drowning out the world with my go-to podcast. \n\nThen, after a long break from that commute, I found myself retracing my old steps. Suddenly, it was as if I had thrown on a pair of fresh glasses! I noticed the vibrant lives aro

In [22]:
def text_to_speech_ai(text, output_file="audio_output.mp3"):
    response = client.audio.speech.create(
        model="gpt-4o-mini-tts",
        voice="alloy",
        input=text
    )
    
    # Save the binary audio stream to a file
    with open(output_file, "wb") as f:
        f.write(response.content)
    print(f"High-quality audio saved to {output_file}")

text_to_speech_ai(res)

High-quality audio saved to audio_output.mp3
